<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/13_AI_Agent/13_02_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13_02 검색증강생성(RAG) : 인덱싱 → 검색 → 생성

**학습 목표**
- **RAG의 필요성**(환각 완화·최신 지식·사내 문서 결합)을 코드로 확인한다.
- **임베딩·의미 검색**으로 질문과 의미가 가까운 문서를 찾는 원리를 이해한다.
- **인덱싱 → 검색 → 생성** 파이프라인을 단계별로 직접 실행해 본다.

> ※ 본 실습은 **개념 소개와 간단한 데모** 수준입니다. 청킹 전략·재순위화(re-ranking)·평가 등 심화 구현은 **'AI 에이전트 개발'** 교과목에서 다룹니다.

> 💡 오픈 웨이트 모델(임베딩·LLM)을 내려받아 실행하므로 **API 키가 필요 없습니다.** 첫 실행은 모델 다운로드로 시간이 걸리니 **런타임 → 런타임 유형 변경 → GPU** 를 권장합니다.

In [31]:
!pip install -q sentence-transformers transformers accelerate torch

## 1. RAG는 왜 필요한가? — 환각(Hallucination) 문제

LLM은 **학습 시점의 지식만** 알고, 모르는 것도 그럴듯한 **거짓(환각)** 으로 답하곤 합니다. 예를 들어 *"우리 회사 최신 환불 규정은?"* 처럼 **사내 문서**나 **최신 정보**가 필요한 질문은 모델의 파라미터만으로는 정확히 답할 수 없습니다.

**RAG(Retrieval-Augmented Generation, 검색증강생성)** 는 질문과 관련된 **외부 문서를 검색해 프롬프트에 근거로 넣어줌**으로써, 모델이 사실에 기반해 답하도록 합니다.

| RAG의 장점 | 설명 |
|------------|------|
| **환각 완화** | 검색된 근거에 기반 → 사실성 ↑ |
| **지식 최신성** | 재학습 없이 최신 문서 반영 |
| **출처 제시** | 근거 문서를 함께 제시 → 신뢰성 ↑ |
| **도메인 적용** | 사내·전문 문서를 지식으로 활용 |

아래에서 **① 근거 없이 답할 때 vs ② 검색된 근거를 넣어줄 때** 의 차이를 직접 만들어 봅니다.

## 2. 임베딩(Embedding)과 의미 검색(Semantic Search)

RAG의 검색은 **키워드 일치가 아니라 의미(semantics)** 로 이루어집니다.

- **임베딩**: 문장을 고정 길이 **벡터**로 변환합니다(7주차 워드 임베딩의 문장 확장판). 의미가 비슷한 문장은 벡터 공간에서 **가깝게** 위치합니다.
- **의미 검색**: 질문 벡터와 문서 벡터의 **코사인 유사도(cosine similarity)** 가 높은 문서를 찾습니다.

먼저 한국어 임베딩 모델로 문장들을 벡터로 바꾸고, **표현은 달라도 의미가 같은 문장**끼리 유사도가 높게 나오는지 확인해 봅니다.

In [32]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("jhgan/ko-sroberta-multitask")  # 한국어 임베딩 모델

sentences = [
    "트랜스포머는 2017년에 발표되었다.",     # A
    "어텐션 논문은 2017년에 나왔다.",         # B (A와 의미 유사, 단어는 다름)
    "오늘 점심은 김치찌개를 먹었다.",         # C (전혀 다른 주제)
]
emb = embedder.encode(sentences, convert_to_tensor=True)
sim = util.cos_sim(emb, emb)  # 문장 쌍별 코사인 유사도 행렬

print("A↔B (의미 유사):", round(sim[0][1].item(), 3))
print("A↔C (의미 무관):", round(sim[0][2].item(), 3))
print("\n벡터 차원:", emb.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

A↔B (의미 유사): 0.637
A↔C (의미 무관): 0.12

벡터 차원: torch.Size([3, 768])


> **관찰 포인트** — A와 B는 겹치는 단어가 거의 없지만 유사도가 높고, 주제가 다른 C와는 낮습니다. 임베딩이 **표면적 단어가 아닌 의미**를 담고 있음을 보여줍니다. 검색은 바로 이 유사도를 이용합니다.

## 3. 인덱싱(Indexing) — 오프라인 단계

지식 베이스를 미리 벡터로 만들어 저장해 두는 **오프라인** 단계입니다.

1. 문서를 적절한 크기로 **청킹(Chunking)** → 2. 각 조각을 **임베딩** → 3. **벡터 DB**에 저장.

> 실무에서는 **FAISS·Chroma·Milvus** 같은 벡터 DB를 쓰지만, 여기서는 원리 이해를 위해 임베딩 텐서(`doc_emb`) 자체를 **벡터 DB 역할**로 사용합니다.

In [33]:
# 지식 베이스(이미 청킹된 문서 조각이라고 가정)
docs = [
    "트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.",
    "BERT는 양방향 인코더로 사전학습된 언어 모델이다.",
    "RAG는 검색한 문서를 프롬프트에 결합해 답을 생성한다.",
    "환불은 구매일로부터 14일 이내에 영수증과 함께 신청해야 한다.",
]

doc_emb = embedder.encode(docs, convert_to_tensor=True)  # 문서 임베딩 = 벡터 DB 역할
print(f"인덱싱 완료: 문서 {len(docs)}개 → 벡터 {tuple(doc_emb.shape)}")

인덱싱 완료: 문서 4개 → 벡터 (4, 768)


## 4. 검색(Retrieval) — 온라인 단계

사용자 질문을 **임베딩**한 뒤, 벡터 DB에서 **가장 유사한 top-k** 문서를 찾습니다. 수백만 벡터 규모에서는 전수 검색 대신 **ANN(근사 최근접 이웃)** 알고리즘으로 빠르게 찾습니다.

In [34]:
def retrieve(query, k=2):
    """질문과 의미가 가까운 문서 top-k를 (문서, 유사도)로 반환한다."""
    q_emb = embedder.encode(query, convert_to_tensor=True)  # 질문 임베딩
    scores = util.cos_sim(q_emb, doc_emb)[0]                # 의미 검색(코사인 유사도)
    top = scores.topk(k)                                    # 상위 k개
    return [(docs[i], round(scores[i].item(), 3)) for i in top.indices]

for doc, score in retrieve("트랜스포머는 언제 나왔나?", k=2):
    print(f"[{score}] {doc}")

[0.435] 트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.
[0.123] BERT는 양방향 인코더로 사전학습된 언어 모델이다.


> 질문에 '2017'이나 '발표'라는 단어가 없어도, **의미가 가장 가까운 문서**가 1순위로 검색됩니다.

## 5. 생성(Generation) — 근거를 프롬프트에 결합

검색된 문서를 **질문과 함께 프롬프트에 결합(augmentation)** 하여 LLM에 전달합니다. 모델은 자신의 기억이 아니라 **주어진 근거**로 답하게 됩니다.

지시학습된 오픈 웨이트 LLM(**Qwen2.5-1.5B-Instruct**)으로 생성 단계까지 실제로 실행합니다.

In [35]:
from transformers import pipeline

llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype="auto", device_map="auto")

def generate(prompt, max_new_tokens=200):
    """프롬프트를 LLM에 전달하고 답변 문자열을 반환한다."""
    out = llm([{"role": "user", "content": prompt}],
              max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()

print("LLM 로드 완료")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

LLM 로드 완료


### 5-1. 근거 없이 답하기 vs 근거를 넣어 답하기

동일한 질문을 **① 근거 없이**, **② 검색된 근거를 넣어** 각각 물어 차이를 비교합니다.

In [36]:
question = "환불은 며칠 이내에 신청해야 하나요?"

# ① 근거 없이 (모델 지식에만 의존 → 환각 위험)
print("=== ① 근거 없음 ===")
print(generate(question, max_new_tokens=80))

# ② 검색된 근거를 결합 (RAG)
context = retrieve(question, k=1)[0][0]
rag_prompt = (
    f"[문서]\n{context}\n\n"
    f"[질문] {question}\n"
    "위 문서를 근거로만 답하라. 문서에 없으면 '모름'이라고 답하라."
)
print("\n=== ② RAG (근거 결합) ===")
print(f"(검색된 근거: {context})\n")
print(generate(rag_prompt, max_new_tokens=80))

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== ① 근거 없음 ===


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


죄송합니다, 저는 실제 서비스를 제공하지 않고 있습니다. 따라서 환불 절차나 시간대에 대한 정보는 제공할 수 없습니다. 그러나 일반적으로 대부분의 온라인 쇼핑몰에서는 7일 내 혹은 최소한 24시간 안에 신청해야 합니다. 하지만 정확한 정보는 해당 사이트의 이용약관이나

=== ② RAG (근거 결합) ===
(검색된 근거: 환불은 구매일로부터 14일 이내에 영수증과 함께 신청해야 한다.)

환불은 구매일로부터 14일 이내에 영수증과 함께 신청해야 합니다.


## 6. 전체 RAG 파이프라인 통합

검색 → 근거 결합 → 생성을 하나의 함수로 묶고, **출처(근거 문서)** 를 함께 보여 줍니다. 지식 베이스에 없는 질문에는 **'모름'** 으로 답하게 하여 환각을 억제합니다.

In [37]:
def rag_answer(question, k=2):
    """검색 → 근거 결합 → 생성. 답변과 출처를 함께 반환한다."""
    hits = retrieve(question, k=k)
    context = "\n".join(f"- {d}" for d, _ in hits)
    prompt = (
        f"[참고 문서]\n{context}\n\n"
        f"[질문] {question}\n"
        "위 문서를 근거로만 간결히 답하라. 문서에 근거가 없으면 '모름'이라고 답하라."
    )
    answer = generate(prompt)
    return answer, [d for d, _ in hits]

for q in ["RAG는 무엇을 결합해 답을 생성하나?",
          "회사 주차장은 몇 층에 있나?"]:   # 지식 베이스에 없는 질문
    ans, src = rag_answer(q, k=2)
    print(f"Q. {q}")
    print(f"A. {ans}")
    print(f"   출처: {src}\n")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q. RAG는 무엇을 결합해 답을 생성하나?
A. RAG는 검색한 문서와 프롬프트를 결합하여 답을 생성한다.
   출처: ['RAG는 검색한 문서를 프롬프트에 결합해 답을 생성한다.', 'BERT는 양방향 인코더로 사전학습된 언어 모델이다.']

Q. 회사 주차장은 몇 층에 있나?
A. 모름
   출처: ['트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.', '환불은 구매일로부터 14일 이내에 영수증과 함께 신청해야 한다.']



## 7. 정리

| 핵심 개념 | 요약 |
|----------|------|
| RAG | 외부 문서 **검색** + LLM **생성** 결합 |
| 임베딩(Embedding) | 텍스트를 고정 길이 벡터로 변환 |
| 벡터 DB | FAISS·Chroma 등 벡터 전용 저장·검색소 |
| 의미 검색 | 코사인 유사도로 의미가 가까운 문서 검색 |
| 인덱싱 / 검색·생성 | 오프라인 준비 / 온라인 응답 두 단계 |

**이번 실습에서 확인한 것**
1. 임베딩은 단어가 아닌 **의미**를 벡터로 담는다.
2. 검색은 질문과 **의미가 가까운** 문서를 top-k로 찾는다.
3. 근거를 결합하면 LLM이 **사실에 기반**해 답하고, 없으면 '모름'으로 환각을 억제한다.

> 💡 **연계 안내**: 청킹 전략·재순위화(re-ranking)·검색 평가·프로덕션 구축 등 RAG 심화는 **'AI 에이전트 개발'** 교과목에서 다룹니다.